In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib, os, numpy as np, pandas as pd
from lightgbm import LGBMRegressor
from scipy.stats import rankdata


def train(X_train, y_train, model_directory_path, loop_moon, embargo):

    feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]

    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    df = df.dropna(subset=["target"])

    # CHANGE 3: cross-sectional median fill (was: fillna(0))
    df[feature_cols] = df.groupby("moon")[feature_cols].transform(
        lambda s: s.fillna(s.median())
    )
    df[feature_cols] = df[feature_cols].fillna(0.0)

    # CHANGE 2: rank-normalise target per moon (was: raw target)
    def cs_rank(s):
        n = len(s)
        if n < 2: return s * 0.0
        r = s.rank(method="average")
        return (r - 1) / (n - 1) - 0.5

    df["target_rank"] = df.groupby("moon")["target"].transform(cs_rank)

    model = LGBMRegressor(
        n_estimators=200, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1
    )
    model.fit(df[feature_cols], df["target_rank"])

    joblib.dump({"model": model, "features": feature_cols},
                os.path.join(model_directory_path, "model.joblib"))
    print(f"  Trained on {X_train['moon'].nunique()} moons, {len(df):,} rows")


In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):

    obj      = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    model    = obj["model"]
    features = obj["features"]

    X = X_test.copy()
    for c in features:
        med = X[c].median()
        X[c] = X[c].fillna(med if not np.isnan(med) else 0.0)

    raw = model.predict(X[features])

    # CHANGE 1: rank-normalise predictions (was: raw output)
    n     = len(raw)
    final = (rankdata(raw) - 1) / max(n - 1, 1) - 0.5

    return pd.DataFrame({
        "moon": X_test["moon"].values,
        "id":   X_test["id"].values,
        "prediction": final,
    })


In [ ]:
from scipy.stats import pearsonr
import os

embargo = 4
model_dir = "./model_step1"
os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []
for moon in splits["reduced_cloud"]:
    pred = infer(X_test_cloud[X_test_cloud["moon"] == moon],
                 model_dir, loop_moon=moon, embargo=embargo)
    results.append(pred)

predictions = pd.concat(results)

print("Step 1 results (preprocessing fixes only):")
corrs = []
for moon in splits["reduced_cloud"]:
    merged = predictions[predictions["moon"] == moon].merge(
        y_test_cloud[y_test_cloud["moon"] == moon], on=["moon", "id"])
    if len(merged) > 1:
        r, _ = pearsonr(merged["prediction"], merged["target"])
        corrs.append(r)
        print(f"  Moon {moon}: Pearson r = {r:.4f}")

import numpy as np
print(f"\nMean IC: {np.mean(corrs):.4f}  (baseline was ~0.0200)")
print("Changes applied: rank-norm predictions + rank-norm target + CS median fill")
